# Question 1 - Clean Training Records Before Dataset Preparation

In [22]:
import math

def clean_training_records(records):
    if not isinstance(records, list):
        return {"cleaned": [], "rejected_indices": []}

    cleaned = []
    rejected_indices = []

    for i, record in enumerate(records):
        if not isinstance(record, dict) or 'learner_id' not in record or 'score' not in record:
            rejected_indices.append(i)
            continue

        l_id, score = record['learner_id'], record['score']

        if not isinstance(l_id, str) or not l_id.strip():
            rejected_indices.append(i)
            continue
        
        if isinstance(score, bool):
            rejected_indices.append(i)
            continue
            
        try:
            score_val = float(score)
            if not math.isfinite(score_val) or not (0 <= score_val <= 100):
                raise ValueError
        except (ValueError, TypeError):
            rejected_indices.append(i)
            continue

        cleaned.append({
            "learner_id": l_id.strip(),
            "score": score_val,
            "eligible": score_val >= 60.0
        })

    return {"cleaned": cleaned, "rejected_indices": rejected_indices}

In [23]:
records = [
    {"learner_id": " L01 ", "score": "82.5"},
    {"learner_id": "L02", "score": 60},
    {"learner_id": "L03", "score": None},
    {"learner_id": "   ", "score": 75},
    {"learner_id": "L04", "score": True},
    {"learner_id": "L05", "score": 48.0},
    {"learner_id": "L06", "score": 105},
]

clean_training_records(records)


{'cleaned': [{'learner_id': 'L01', 'score': 82.5, 'eligible': True},
  {'learner_id': 'L02', 'score': 60.0, 'eligible': True},
  {'learner_id': 'L05', 'score': 48.0, 'eligible': False}],
 'rejected_indices': [2, 3, 4, 6]}

In [24]:
#This test verifies the rejection of elements that are not dictionaries, as well as records missing the required learner_id or
#score keys. It also confirms that extra fields are ignored.

test_1 = [
    "not a dictionary", 
    {"learner_id": "L10"}, 
    {"score": 90},         
    {"learner_id": "L11", "score": 90, "extra_key": "ignoreedd"} 
]

clean_training_records(test_1)

{'cleaned': [{'learner_id': 'L11', 'score': 90.0, 'eligible': True}],
 'rejected_indices': [0, 1, 2]}

In [25]:
#This test evaluates the strict constraints on the score field, rejecting Booleans, strings 
# like "nan" and "inf" whose converted values are not finite, and values outside the 0 through 100 range.

test_2 = [
    {"learner_id": "L20", "score": False}, 
    {"learner_id": "L21", "score": "inf"}, 
    {"learner_id": "L22", "score": "nan"}, 
    {"learner_id": "L23", "score": 105},   
    {"learner_id": "L24", "score": -5.5},  
    {"learner_id": "L25", "score": 100.0}  
]

clean_training_records(test_2)


{'cleaned': [{'learner_id': 'L25', 'score': 100.0, 'eligible': True}],
 'rejected_indices': [0, 1, 2, 3, 4]}

In [26]:
#This test ensures that the learner_id is only accepted when it is a string that remains non-empty after trimming surrounding whitespace.

test_3 = [
    {"learner_id": None, "score": 50},
    {"learner_id": 123, "score": 50},
    {"learner_id": "   ", "score": 50}, 
    {"learner_id": "\t\n", "score": 50},
    {"learner_id": " L30 ", "score": 50} 
]

clean_training_records(test_3)

{'cleaned': [{'learner_id': 'L30', 'score': 50.0, 'eligible': False}],
 'rejected_indices': [0, 1, 2, 3]}

### Why does checking only whether a score is an integer risk accepting Boolean values?

### Ans . Because in Python, the Boolean values `True` and `False` are subclasses of the integer type.
### Specifically, `True` is equivalent to the integer `1`, and `False` is equivalent to the integer `0`. Therefore, if you only check whether a score is an integer, you would inadvertently accept Boolean values as valid scores, which may not be the intended behavior for your application.

# Question 2 - Prepare Mixed Text Payloads for an NLP Pipeline


In [27]:
def prepare_messages(payloads, min_length=3):
    # Enforce min_length as a positive integer strictly excluding Booleans
    if type(min_length) is not int or min_length <= 0:
        raise ValueError("min_length must be a positive integer excluding Booleans.")

    token_lists = []
    rejected_indices = []
    strip_chars = ".,!?;:"

    for i, payload in enumerate(payloads):
        # 1. Type validation and Decoding
        if isinstance(payload, str):
            text = payload
        elif isinstance(payload, (bytes, bytearray)):
            try:
                text = payload.decode('utf-8')
            except UnicodeDecodeError:
                rejected_indices.append(i)
                continue
        else:
            rejected_indices.append(i)
            continue

        # 2. Processing: lowercase, split, strip punctuation, and filter by length
        # (Uses the required list comprehension)
        tokens = [
            word.strip(strip_chars) for word in text.lower().split()
            if len(word.strip(strip_chars)) >= min_length
        ]
        token_lists.append(tokens)

    # 3. Vocabulary Construction
    # (Uses the required set comprehension to find unique tokens across all lists)
    vocabulary = sorted({token for lst in token_lists for token in lst})

    return {
        "token_lists": token_lists,
        "vocabulary": vocabulary,
        "rejected_indices": rejected_indices
    }

In [28]:
payloads_sample = [
    "  AI, helps TEAMS! ",
    b"Build reliable agents.",
    bytearray(b"AI helps."),
    None,
    b"\xff",
]
print("--- Sample Test ---")
print(prepare_messages(payloads_sample))
print()

# Test 1:"agent-based" should remain untouched inside, and length 3 words should stay.
test_1 = ["agent-based systems.", "a, b, c!"]
print("--- Test 1: Internal Punctuation ---")
print(prepare_messages(test_1, min_length=3))
print()

# Test 2: Empty inputs and whitespace Should produce empty token lists without rejecting the payload.
test_2 = ["", "   \t\n  ", b""]
print("--- Test 2: Whitespace and Empty Strings ---")
print(prepare_messages(test_2))
print()

# Test 3: Passing a Boolean or negative number must raise a ValueError.
print("--- Test 3: Invalid min_length ---")
try:
    prepare_messages(["valid string"], min_length=True)
    print("Failed: Should have raised ValueError")
except ValueError as e:
    print(f"Successfully caught ValueError: {e}")

--- Sample Test ---
{'token_lists': [['helps', 'teams'], ['build', 'reliable', 'agents'], ['helps']], 'vocabulary': ['agents', 'build', 'helps', 'reliable', 'teams'], 'rejected_indices': [3, 4]}

--- Test 1: Internal Punctuation ---
{'token_lists': [['agent-based', 'systems'], []], 'vocabulary': ['agent-based', 'systems'], 'rejected_indices': []}

--- Test 2: Whitespace and Empty Strings ---
{'token_lists': [[], [], []], 'vocabulary': [], 'rejected_indices': []}

--- Test 3: Invalid min_length ---
Successfully caught ValueError: min_length must be a positive integer excluding Booleans.


## What practical difference between bytes and bytearray matters when processing input without modifying it?  

## The practical difference between `bytes` and `bytearray`that matters when processing input without modifying it is that `bytes` is immutable, while `bytearray` is mutable. 

## When you are processing input data without the intention of modifying it, using `bytes` is generally preferred because it ensures that the data cannot be changed accidentally during processing. This immutability can help prevent bugs and maintain data integrity.

# Question 3: Summarise Classification Prediction Results

In [29]:
def summarise_predictions(results):
    valid_count = 0
    correct_count = 0
    by_actual = {}
    misclassified_ids = []
    rejected_indices = []

    if not isinstance(results, list):
        return {"valid_count": 0, "correct_count": 0, "accuracy": None, 
                "by_actual": {}, "misclassified_ids": [], "rejected_indices": []}

    for i, record in enumerate(results):
        # Reject non-dictionaries and missing keys
        if not isinstance(record, dict) or not all(k in record for k in ("id", "actual", "predicted")):
            rejected_indices.append(i)
            continue

        rec_id, actual, predicted = record["id"], record["actual"], record["predicted"]

        # Reject non-strings or completely blank strings
        if not (isinstance(rec_id, str) and isinstance(actual, str) and isinstance(predicted, str)):
            rejected_indices.append(i)
            continue
        if not (rec_id.strip() and actual.strip() and predicted.strip()):
            rejected_indices.append(i)
            continue

        # Processing valid records
        valid_count += 1
        act_clean = actual.strip().lower()
        pred_clean = predicted.strip().lower()

        # Initialise category if not seen
        if act_clean not in by_actual:
            by_actual[act_clean] = {"total": 0, "correct": 0}

        by_actual[act_clean]["total"] += 1

        # Check correctness
        if act_clean == pred_clean:
            correct_count += 1
            by_actual[act_clean]["correct"] += 1
        else:
            misclassified_ids.append(rec_id)

    # Calculate accuracy
    accuracy = round(correct_count / valid_count, 4) if valid_count > 0 else None

    return {
        "valid_count": valid_count,
        "correct_count": correct_count,
        "accuracy": accuracy,
        "by_actual": by_actual,
        "misclassified_ids": misclassified_ids,
        "rejected_indices": rejected_indices
    }

In [31]:
results_sample = [
    {"id": "T1", "actual": "Billing", "predicted": " billing "},
    {"id": "T2", "actual": "Technical", "predicted": "billing"},
    {"id": "T3", "actual": "billing", "predicted": "billing"},
    {"id": "T4", "actual": "Account", "predicted": "ACCOUNT"},
    {"id": "T5", "actual": None, "predicted": "account"},
]
print(summarise_predictions(results_sample))
print()

print("---Test 1: Empty and Invalid Inputs ---")
print(summarise_predictions([[], "string", {"id": "T1", "actual": "   ", "predicted": "x"}]))
print()

print("---Test 2: Extra fields and Repeated Records ---")
# Repeated records should count separately, extra fields ignored
test_repeat = [
    {"id": "T1", "actual": "bug", "predicted": "bug", "extra": 123},
    {"id": "T1", "actual": "bug", "predicted": "bug", "ignore": "me"}
]
print(summarise_predictions(test_repeat))

{'valid_count': 4, 'correct_count': 3, 'accuracy': 0.75, 'by_actual': {'billing': {'total': 2, 'correct': 2}, 'technical': {'total': 1, 'correct': 0}, 'account': {'total': 1, 'correct': 1}}, 'misclassified_ids': ['T2'], 'rejected_indices': [4]}

---Test 1: Empty and Invalid Inputs ---
{'valid_count': 0, 'correct_count': 0, 'accuracy': None, 'by_actual': {}, 'misclassified_ids': [], 'rejected_indices': [0, 1, 2]}

---Test 2: Extra fields and Repeated Records ---
{'valid_count': 2, 'correct_count': 2, 'accuracy': 1.0, 'by_actual': {'bug': {'total': 2, 'correct': 2}}, 'misclassified_ids': [], 'rejected_indices': []}


## Interview Follow-up: Why is None more informative than 0.0 when accuracy cannot be calculated?

## 0.0 implies a calculated mathematical result where the system attempted classifications and got 0% of them correct. None correctly represents the absence of data—indicating that no valid predictions were evaluated at all, preventing engineers from mistakenly believing the model failed its classification task.

## Question 4: Detect Dataset Duplicates and Overlap

In [32]:
def audit_dataset_ids(train_ids, validation_ids):
    def process_ids(id_list):
        valid = []
        invalid_idx = []
        # Return empty lists if not given a list
        if not isinstance(id_list, list):
            return valid, invalid_idx
            
        for i, val in enumerate(id_list):
            if not isinstance(val, str) or not val.strip():
                invalid_idx.append(i)
            else:
                valid.append(val.strip())
        return valid, invalid_idx

    def get_duplicates_and_unique(valid_list):
        seen = set()
        duplicates = set()
        for item in valid_list:
            if item in seen:
                duplicates.add(item)
            seen.add(item)
        return sorted(list(duplicates)), frozenset(seen)

    t_valid, t_invalid = process_ids(train_ids)
    v_valid, v_invalid = process_ids(validation_ids)

    t_dups, t_frozenset = get_duplicates_and_unique(t_valid)
    v_dups, v_frozenset = get_duplicates_and_unique(v_valid)

    # Set operations for overlap and exclusive items
    overlap = sorted(list(t_frozenset & v_frozenset))
    train_only = sorted(list(t_frozenset - v_frozenset))
    val_only = sorted(list(v_frozenset - t_frozenset))

    return {
        "train_duplicates": t_dups,
        "validation_duplicates": v_dups,
        "overlap": overlap,
        "train_only": train_only,
        "validation_only": val_only,
        "invalid_train_indices": t_invalid,
        "invalid_validation_indices": v_invalid,
        "snapshots": {
            "train": t_frozenset,
            "validation": v_frozenset
        }
    }

In [35]:
train_ids_sample = ["R3", "R1", " R2 ", "R1", None]
validation_ids_sample = ["R2", "R4", "R4", " "]
print(audit_dataset_ids(train_ids_sample, validation_ids_sample))
print()

print("---Test 1: No overlaps, no duplicates ---")
print(audit_dataset_ids(["A1", "A2"], ["B1", "B2"]))
print()

print("---Test 2: Case sensitivity and exact duplicates ---")
# "a1" and "A1" are different. Multiple duplicates should only list once.
print(audit_dataset_ids(["A1", "a1", "A1", "A1"], ["a1", "a1"]))
print()

{'train_duplicates': ['R1'], 'validation_duplicates': ['R4'], 'overlap': ['R2'], 'train_only': ['R1', 'R3'], 'validation_only': ['R4'], 'invalid_train_indices': [4], 'invalid_validation_indices': [3], 'snapshots': {'train': frozenset({'R3', 'R2', 'R1'}), 'validation': frozenset({'R4', 'R2'})}}

---Test 1: No overlaps, no duplicates ---
{'train_duplicates': [], 'validation_duplicates': [], 'overlap': [], 'train_only': ['A1', 'A2'], 'validation_only': ['B1', 'B2'], 'invalid_train_indices': [], 'invalid_validation_indices': [], 'snapshots': {'train': frozenset({'A2', 'A1'}), 'validation': frozenset({'B2', 'B1'})}}

---Test 2: Case sensitivity and exact duplicates ---
{'train_duplicates': ['A1'], 'validation_duplicates': ['a1'], 'overlap': ['a1'], 'train_only': ['A1'], 'validation_only': [], 'invalid_train_indices': [], 'invalid_validation_indices': [], 'snapshots': {'train': frozenset({'a1', 'A1'}), 'validation': frozenset({'a1'})}}



## Interview Follow-up: What information is lost when dataset identifiers are converted directly into a set?

## Converting a list directly to a set permanently deletes three pieces of context: the original order of the identifiers, the frequency/count of any repeated identifiers (duplicates are instantly merged), and the ability to index directly into the collection.

## Question 5: Update an Annotation Queue

In [36]:
def update_annotation_queue(queue, incoming, completed, priority=None, batch_size=3):
    # Enforce batch_size constraints strictly
    if type(batch_size) is not int or batch_size < 0:
        raise ValueError("batch_size must be a non-negative integer.")

    # 1. Combine lists retaining first occurrence without modifying originals
    combined_queue = []
    seen = set()
    for item in queue + incoming:
        if item not in seen:
            combined_queue.append(item)
            seen.add(item)

    # 2. Remove completed
    completed_set = set(completed)
    updated_queue = []
    for item in combined_queue:
        if item not in completed_set:
            updated_queue.append(item)

    # 3. Move priority to front (List method 1: .remove(), List method 2: .insert())
    if priority in updated_queue:
        updated_queue.remove(priority)
        updated_queue.insert(0, priority)

    # 4. Generate outputs 
    next_batch = updated_queue[:batch_size] # Slicing
    last_item = updated_queue[-1] if updated_queue else None # Indexing

    positions = []
    for i in range(len(updated_queue)): # Range
        positions.append((i + 1, updated_queue[i]))

    return {
        "queue": updated_queue,
        "next_batch": next_batch,
        "last_item": last_item,
        "positions": positions
    }

In [38]:
queue_sample = ["D3", "D1", "D3", "D2"]
incoming_sample = ["D4", "D2", "D5"]
completed_sample = ["D1", "D9"]
print(update_annotation_queue(
    queue_sample, incoming_sample, completed_sample,
    priority="D5", batch_size=3,
))
print()

print("---Test 1: batch_size = 0 and priority not in queue ---")
print(update_annotation_queue(
    ["D1"], ["D2"], ["D3"], priority="D99", batch_size=0
))
print()

print("---Test 2: All items completed (Empty output handling) ---")
print(update_annotation_queue(
    ["D1", "D2"], [], ["D1", "D2"], batch_size=5
))
print()

{'queue': ['D5', 'D3', 'D2', 'D4'], 'next_batch': ['D5', 'D3', 'D2'], 'last_item': 'D4', 'positions': [(1, 'D5'), (2, 'D3'), (3, 'D2'), (4, 'D4')]}

---Test 1: batch_size = 0 and priority not in queue ---
{'queue': ['D1', 'D2'], 'next_batch': [], 'last_item': 'D2', 'positions': [(1, 'D1'), (2, 'D2')]}

---Test 2: All items completed (Empty output handling) ---
{'queue': [], 'next_batch': [], 'last_item': None, 'positions': []}



## Interview Follow-up: Why would assigning the original queue to another variable fail to create an independent queue?

## In Python, variables are essentially "name tags" attached to objects in memory. If you write new_queue = queue, you aren't copying the list; you are just attaching a second name tag to the exact same list in memory. If you use .append() or .remove() on new_queue, it mutates the underlying object, meaning the original queue will also be altered, breaking the isolation constraint.